# Step 4: Fine-Tuning with QLoRA

Fine-tune a Llama 3.1 8B model using QLoRA (4-bit quantized LoRA) via Unsloth.

**What this notebook covers:**
- Loading base model with 4-bit quantization
- Configuring LoRA adapters
- Setting up SFTTrainer with WandB logging
- Running training and monitoring loss
- Saving adapter weights

**Requirements:** GPU with >= 16GB VRAM (A100, H100, or RTX 4090)

# ⚠️ CRITICAL WARNING - READ BEFORE RUNNING

**This notebook will RE-TRAIN YOUR MODEL and OVERWRITE existing files.**

## Purpose
This is an **educational walkthrough** demonstrating how QLoRA training works. It will:
- Train a NEW model from scratch (10-60 minutes of GPU time)
- OVERWRITE `models/sql-llama-8b-lora/` if it exists
- **LOSE YOUR CURRENTLY TRAINED MODEL**

## When to Use This Notebook
✅ **LEARNING**: Understanding how QLoRA training works  
✅ **DEVELOPMENT**: Testing new hyperparameters  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **EVALUATION**: You have already trained a model and want to evaluate it  
❌ **PRODUCTION**: You want to use your existing trained model  
❌ **COMPARISON**: You want to compare teacher vs student models

## What You Should Run Instead
If you have completed training and want to evaluate your model, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

## Current Status
Your trained model exists at: `models/sql-llama-8b-lora/`
- Adapter weights: 161 MB
- Training completed: 2026-04-18
- **Running this notebook will DELETE it!**

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 4: Fine-Tuning with QLoRA

Fine-tune a Llama 3.1 8B model using QLoRA (4-bit quantized LoRA) via Unsloth.

**What this notebook covers:**
- Loading base model with 4-bit quantization
- Configuring LoRA adapters
- Setting up SFTTrainer with WandB logging
- Running training and monitoring loss
- Saving adapter weights

**Requirements:** GPU with >= 16GB VRAM (A100, H100, or RTX 4090)

In [ ]:
import sys
sys.path.insert(0, '..')

from datasets import load_dataset
from src.train.lora import build_lora_model, LoRAConfig, get_default_config
from src.train.runner import TrainingRunner
from src.train.monitor import TrainingMonitor

In [ ]:
# Load base model with QLoRA
lora_config = get_default_config('llama')
print(f'LoRA config: rank={lora_config.rank}, alpha={lora_config.alpha}')

model, tokenizer = build_lora_model(
    base_model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    lora_config=lora_config
)

In [ ]:
# Load formatted dataset
dataset = load_dataset('json', data_files={
    'train': 'data/formatted/train.jsonl',
    'validation': 'data/formatted/val.jsonl'
})
print(dataset)

In [ ]:
# Setup and run training
runner = TrainingRunner(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    val_dataset=dataset['validation'],
)

runner.train()

In [ ]:
# Check for overfitting
monitor = TrainingMonitor()
monitor.log_gpu_stats()

In [ ]:
# Save adapter
runner.save('models/sql-llama-8b-lora')
print('LoRA adapter saved to models/sql-llama-8b-lora/')